# Predicción de Miopía en Estudiantes de Medicina
## Universidad de Navarra

### Objetivo:
Predecir 4 variables relacionadas con la miopía:
1. **M**: Clasificación binaria (SI/NO) - ¿Tiene miopía?
2. **MM**: Clasificación binaria (SI/NO) - ¿Tiene miopía magna?
3. **COMBO**: Clasificación multi-clase (M, MM, C) - Tipo de condición
4. **DCOMBO**: Clasificación multi-clase (M1, M2, MM, C) - Clasificación detallada

## 1. Importar Librerías Necesarias

In [1]:
# Librerías para manejo de datos
import pandas as pd
import numpy as np

# Librerías para preprocesamiento
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Librerías para modelado
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Librerías para validación
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Configuración para ignorar warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 2. Cargar los Datos

In [2]:
# Cargar los conjuntos de datos
X_train = pd.read_csv('X_train.csv')  # Características de entrenamiento
Y_train = pd.read_csv('Y_train.csv')  # Etiquetas de entrenamiento
X_test = pd.read_csv('X_test.csv')    # Características de test

# Mostrar dimensiones de los datos
print(f"Dimensiones del training set: {X_train.shape}")
print(f"Dimensiones del test set: {X_test.shape}")
print(f"Número de etiquetas a predecir: {Y_train.shape[1]}")

# Mostrar las primeras filas
print("\n📊 Primeras filas del conjunto de entrenamiento:")
display(X_train.head())

print("\n🎯 Etiquetas del conjunto de entrenamiento:")
display(Y_train.head())

Dimensiones del training set: (132, 49)
Dimensiones del test set: (24, 49)
Número de etiquetas a predecir: 4

📊 Primeras filas del conjunto de entrenamiento:


,fecha,edad,sexo,origen antepasados (españa),origen antepasados (extranjeros),hº deporte sem,hº naturaleza sem,hº ocio exteriores sem,horas exterior sem,hº estudiar sem,...,OS-DS,CUVAF-Total,CUVAF-Media,CUVAF-DS,Promedio Nasal,DE Nasal,Suma Nasal,Promedio Temporal,DE Temporal,Suma Temporal
0,2/11/20,22,Mujer,Asturias,NaN,0.0,0.0,4.0,4.0,40,...,0.238059,2.256667,0.564167,0.842300,0.905000,1.279863,1.810000,0.223333,0.160278,0.446667
1,2/17/20,23,Mujer,Navarra,NaN,5.0,2.0,2.0,9.0,20,...,2.585654,8.026667,2.006667,1.887316,3.620833,0.503224,7.241667,0.392500,0.102530,0.785000
2,3/10/20,22,Mujer,Bizkaia,NaN,2.0,3.0,8.0,13.0,30,...,0.431335,12.710000,3.177500,0.743668,3.152500,1.248043,6.305000,3.202500,0.314663,6.405000
3,2/26/20,22,Mujer,"Navarra, Extremadura, Cataluña, Aragón",España,0.0,2.0,13.0,15.0,54,...,0.626968,3.551667,0.887917,0.877145,1.003333,1.418928,2.006667,0.772500,0.491439,1.545000
4,2/19/20,23,Mujer,Toledo,NaN,3.0,0.0,0.0,3.0,56,...,1.269257,1.795000,0.448750,0.897500,0.897500,1.269257,1.795000,0.000000,0.000000,0.000000



🎯 Etiquetas del conjunto de entrenamiento:


,M,MM,Combo,DCombo
0,SI,SI,MM,MM
1,SI,NO,M,M1
2,NO,NO,C,C
3,NO,NO,C,C
4,NO,NO,C,C


## 3. Análisis Exploratorio de las Etiquetas

In [3]:
# Mostrar la distribución de cada etiqueta en el conjunto de entrenamiento
print("📊 Distribución de clases en el conjunto de entrenamiento:\n")

for col in Y_train.columns:
    print(f"\n{col}:")
    print(Y_train[col].value_counts())
    print("-" * 50)

📊 Distribución de clases en el conjunto de entrenamiento:


M:
M
SI    88
NO    44
Name: count, dtype: int64
--------------------------------------------------

MM:
MM
NO    122
SI     10
Name: count, dtype: int64
--------------------------------------------------

Combo:
Combo
M     70
C     44
MM    18
Name: count, dtype: int64
--------------------------------------------------

DCombo:
DCombo
C     44
M1    39
M2    31
MM    18
Name: count, dtype: int64
--------------------------------------------------


## 4. Preprocesamiento de Datos

### 4.1 Función de Preprocesamiento

In [4]:
def preprocess_data(X, label_encoders=None, is_training=True):
    """
    Preprocesa el conjunto de datos:
    - Elimina la columna de fecha (no es útil para predicción)
    - Imputa valores faltantes en variables numéricas con la mediana
    - Codifica variables categóricas usando LabelEncoder
    - Maneja valores infinitos reemplazándolos por 0
    
    Parámetros:
    -----------
    X : DataFrame
        Conjunto de datos a preprocesar
    label_encoders : dict, optional
        Diccionario con los encoders ya entrenados (para test set)
    is_training : bool
        Si True, entrena nuevos encoders. Si False, usa los existentes.
    
    Retorna:
    --------
    X_processed : DataFrame
        Datos preprocesados
    label_encoders : dict
        Diccionario con los encoders entrenados
    """
    X_proc = X.copy()
    
    # 1. Eliminar columna de fecha si existe (no aporta información predictiva)
    if 'fecha' in X_proc.columns:
        X_proc = X_proc.drop('fecha', axis=1)
        print("   ✓ Columna 'fecha' eliminada")
    
    # 2. Identificar tipos de columnas
    numeric_cols = X_proc.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X_proc.select_dtypes(include=['object']).columns.tolist()
    
    print(f"   ✓ Columnas numéricas: {len(numeric_cols)}")
    print(f"   ✓ Columnas categóricas: {len(categorical_cols)}")
    
    # 3. Imputar valores faltantes en columnas numéricas
    # Usamos la mediana porque es robusta a outliers
    for col in numeric_cols:
        if X_proc[col].isnull().sum() > 0:
            median_val = X_proc[col].median()
            # Si todos los valores son NaN, usar 0
            if pd.isna(median_val):
                median_val = 0
            X_proc[col].fillna(median_val, inplace=True)
        
        # Reemplazar valores infinitos por 0
        X_proc[col].replace([np.inf, -np.inf], 0, inplace=True)
    
    # 4. Codificar variables categóricas
    if is_training:
        # Modo entrenamiento: crear nuevos encoders
        label_encoders = {}
        for col in categorical_cols:
            # Rellenar valores faltantes con 'Unknown'
            X_proc[col].fillna('Unknown', inplace=True)
            
            # Crear y entrenar el encoder
            le = LabelEncoder()
            X_proc[col] = le.fit_transform(X_proc[col].astype(str))
            label_encoders[col] = le
    else:
        # Modo test: usar encoders existentes
        for col in categorical_cols:
            X_proc[col].fillna('Unknown', inplace=True)
            
            if col in label_encoders:
                le = label_encoders[col]
                # Manejar categorías no vistas en training: asignar -1
                X_proc[col] = X_proc[col].apply(
                    lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else -1
                )
            else:
                # Si no hay encoder para esta columna, crear uno nuevo
                le = LabelEncoder()
                X_proc[col] = le.fit_transform(X_proc[col].astype(str))
    
    # 5. Limpieza final: asegurar que no hay NaN ni Inf
    X_proc = X_proc.fillna(0)
    numeric_cols_final = X_proc.select_dtypes(include=[np.number]).columns
    for col in numeric_cols_final:
        X_proc[col] = X_proc[col].replace([np.inf, -np.inf], 0)
    
    print("   ✓ Preprocesamiento completado")
    
    return X_proc, label_encoders

### 4.2 Aplicar Preprocesamiento

In [5]:
print("📝 Preprocesando conjunto de ENTRENAMIENTO...")
X_train_proc, label_encoders = preprocess_data(X_train, is_training=True)

print("\n📝 Preprocesando conjunto de TEST...")
X_test_proc, _ = preprocess_data(X_test, label_encoders=label_encoders, is_training=False)

print(f"\n✅ Forma final del training: {X_train_proc.shape}")
print(f"✅ Forma final del test: {X_test_proc.shape}")

📝 Preprocesando conjunto de ENTRENAMIENTO...
   ✓ Columna 'fecha' eliminada
   ✓ Columnas numéricas: 33
   ✓ Columnas categóricas: 15
   ✓ Preprocesamiento completado

📝 Preprocesando conjunto de TEST...
   ✓ Columna 'fecha' eliminada
   ✓ Columnas numéricas: 33
   ✓ Columnas categóricas: 15
   ✓ Preprocesamiento completado

✅ Forma final del training: (132, 48)
✅ Forma final del test: (24, 48)


### 4.3 Estandarización de Características

La estandarización es importante para que todas las características tengan la misma escala,
especialmente para algoritmos sensibles a la escala como SVM y Logistic Regression.

In [6]:
# Crear el estandarizador
scaler = StandardScaler()

# Entrenar con datos de training y transformar
X_train_scaled = scaler.fit_transform(X_train_proc)

# Solo transformar (no entrenar) con datos de test
X_test_scaled = scaler.transform(X_test_proc)

print("✅ Características estandarizadas (media=0, std=1)")
print(f"   Media del training después de escalar: {X_train_scaled.mean():.6f}")
print(f"   Desviación estándar del training: {X_train_scaled.std():.6f}")

✅ Características estandarizadas (media=0, std=1)
   Media del training después de escalar: 0.000000
   Desviación estándar del training: 0.989529


## 5. Entrenamiento de Modelos

### 5.1 Función para Entrenar y Seleccionar el Mejor Modelo

In [7]:
def train_best_model(X_train, y_train, label_name):
    """
    Entrena múltiples modelos y selecciona el mejor basándose en validación cruzada.
    
    Se evalúan 4 algoritmos diferentes:
    - Random Forest: Ensemble de árboles de decisión
    - Gradient Boosting: Boosting secuencial de árboles
    - SVM: Support Vector Machine con kernel RBF
    - Logistic Regression: Modelo lineal con regularización L2
    
    La selección se hace mediante validación cruzada estratificada (5-fold)
    para mantener la proporción de clases en cada fold.
    
    Los modelos usan class_weight='balanced' para manejar automáticamente
    clases desbalanceadas dando más peso a las clases minoritarias.
    
    Parámetros:
    -----------
    X_train : array
        Características de entrenamiento
    y_train : array
        Etiquetas de entrenamiento
    label_name : str
        Nombre de la etiqueta (para logging)
    
    Retorna:
    --------
    best_model : estimator
        Modelo entrenado con mejor rendimiento
    """
    print(f"\n🎯 Entrenando modelos para: {label_name}")
    print("-" * 60)
    
    # Definir los modelos a evaluar con sus hiperparámetros
    # Usamos class_weight='balanced' para manejar clases desbalanceadas
    models = {
        'Random Forest': RandomForestClassifier(
            n_estimators=200,        # Número de árboles
            max_depth=10,            # Profundidad máxima
            min_samples_split=5,     # Mínimo de muestras para dividir
            min_samples_leaf=2,      # Mínimo de muestras en hoja
            class_weight='balanced', # Ajustar pesos por desbalanceo
            random_state=42,
            n_jobs=-1                # Usar todos los cores
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=150,        # Número de boosting stages
            learning_rate=0.1,       # Tasa de aprendizaje
            max_depth=5,             # Profundidad de cada árbol
            random_state=42
        ),
        'SVM': SVC(
            kernel='rbf',            # Kernel Radial Basis Function
            C=1.0,                   # Parámetro de regularización
            gamma='scale',           # Coeficiente del kernel
            class_weight='balanced', # Ajustar pesos por desbalanceo
            random_state=42
        ),
        'Logistic Regression': LogisticRegression(
            C=1.0,                   # Inverso de la fuerza de regularización
            max_iter=1000,           # Máximo de iteraciones
            class_weight='balanced', # Ajustar pesos por desbalanceo
            random_state=42
        )
    }
    
    # Configurar validación cruzada estratificada
    # Estratificada significa que mantiene la proporción de clases en cada fold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Evaluar cada modelo
    results = {}
    for name, model in models.items():
        # Realizar validación cruzada
        scores = cross_val_score(
            model, X_train, y_train, 
            cv=cv, 
            scoring='accuracy',  # Métrica: porcentaje de aciertos
            n_jobs=-1
        )
        
        # Guardar resultados
        results[name] = {
            'mean': scores.mean(),
            'std': scores.std(),
            'model': model
        }
        
        print(f"   {name:20s}: {scores.mean():.4f} (+/- {scores.std():.4f})")
    
    # Seleccionar el mejor modelo (mayor accuracy promedio)
    best_model_name = max(results, key=lambda x: results[x]['mean'])
    best_model = results[best_model_name]['model']
    best_score = results[best_model_name]['mean']
    
    print(f"\n   ✨ Mejor modelo: {best_model_name}")
    print(f"   📊 Accuracy: {best_score:.4f}")
    
    # Entrenar el mejor modelo con todos los datos de entrenamiento
    best_model.fit(X_train, y_train)
    
    return best_model

### 5.2 Entrenar Modelos para Cada Etiqueta

In [8]:
print("="*80)
print("ENTRENAMIENTO DE MODELOS PREDICTIVOS")
print("="*80)

# Diccionario para almacenar los modelos entrenados
models = {}

# Entrenar un modelo para cada etiqueta
for label in Y_train.columns:
    print(f"\n{'='*80}")
    
    # Entrenar y seleccionar el mejor modelo
    best_model = train_best_model(
        X_train_scaled,
        Y_train[label].values,
        label
    )
    
    # Guardar el modelo entrenado
    models[label] = best_model

print(f"\n{'='*80}")
print("✅ Entrenamiento completado para todas las etiquetas")
print("="*80)

ENTRENAMIENTO DE MODELOS PREDICTIVOS


🎯 Entrenando modelos para: M
------------------------------------------------------------
   Random Forest       : 0.7647 (+/- 0.0580)
   Gradient Boosting   : 0.7194 (+/- 0.0519)
   SVM                 : 0.7647 (+/- 0.0519)
   Logistic Regression : 0.8026 (+/- 0.0672)

   ✨ Mejor modelo: Logistic Regression
   📊 Accuracy: 0.8026


🎯 Entrenando modelos para: MM
------------------------------------------------------------
   Random Forest       : 0.9242 (+/- 0.0014)
   Gradient Boosting   : 0.9322 (+/- 0.0439)
   SVM                 : 0.9242 (+/- 0.0244)
   Logistic Regression : 0.9165 (+/- 0.0284)

   ✨ Mejor modelo: Gradient Boosting
   📊 Accuracy: 0.9322


🎯 Entrenando modelos para: Combo
------------------------------------------------------------
   Random Forest       : 0.7336 (+/- 0.1106)
   Gradient Boosting   : 0.7043 (+/- 0.0897)
   SVM                 : 0.6738 (+/- 0.0727)
   Logistic Regression : 0.7490 (+/- 0.0603)

   ✨ Mejor modelo: 

## 6. Predicciones en el Conjunto de Test

In [9]:
print("\n🔮 Generando predicciones para el conjunto de test...\n")

# Diccionario para almacenar las predicciones
predictions = {}

# Predecir cada etiqueta usando su modelo entrenado
for label in Y_train.columns:
    model = models[label]
    pred = model.predict(X_test_scaled)
    predictions[label] = pred
    print(f"   ✓ Predicciones generadas para {label}")

# Crear DataFrame con las predicciones
predictions_df = pd.DataFrame(predictions)

print("\n✅ Todas las predicciones generadas exitosamente")


🔮 Generando predicciones para el conjunto de test...

   ✓ Predicciones generadas para M
   ✓ Predicciones generadas para MM
   ✓ Predicciones generadas para Combo
   ✓ Predicciones generadas para DCombo

✅ Todas las predicciones generadas exitosamente


## 7. Visualizar las Predicciones

In [10]:
# Mostrar las predicciones completas
print("\n📋 PREDICCIONES COMPLETAS PARA EL CONJUNTO DE TEST:")
print("="*80)
display(predictions_df)

# Mostrar distribución de las predicciones
print("\n📊 DISTRIBUCIÓN DE LAS PREDICCIONES:")
print("="*80)

for col in predictions_df.columns:
    print(f"\n{col}:")
    print(predictions_df[col].value_counts())
    print("-" * 40)


📋 PREDICCIONES COMPLETAS PARA EL CONJUNTO DE TEST:


,M,MM,Combo,DCombo
0,SI,NO,M,MM
1,NO,NO,C,M1
2,SI,SI,MM,MM
3,SI,NO,M,M1
4,SI,NO,M,C
5,SI,NO,M,M1
6,SI,NO,M,M1
7,SI,SI,MM,MM
8,SI,NO,M,M1
9,SI,NO,MM,MM



📊 DISTRIBUCIÓN DE LAS PREDICCIONES:

M:
M
SI    17
NO     7
Name: count, dtype: int64
----------------------------------------

MM:
MM
NO    22
SI     2
Name: count, dtype: int64
----------------------------------------

Combo:
Combo
M     13
C      7
MM     4
Name: count, dtype: int64
----------------------------------------

DCombo:
DCombo
M1    9
C     6
MM    5
M2    4
Name: count, dtype: int64
----------------------------------------


## 8. Resumen Final

In [11]:
# Crear resumen estadístico
print("\n" + "="*80)
print("RESUMEN DEL ANÁLISIS")
print("="*80)

print(f"\n📊 Dataset:")
print(f"   • Muestras de entrenamiento: {len(X_train)}")
print(f"   • Muestras de test: {len(X_test)}")
print(f"   • Número de características: {X_train_proc.shape[1]}")
print(f"   • Etiquetas predichas: {len(predictions_df.columns)}")

print(f"\n🎯 Predicciones generadas:")
print(f"   • M (Miopía):")
for clase, count in predictions_df['M'].value_counts().items():
    print(f"     - {clase}: {count} estudiantes ({count/len(predictions_df)*100:.1f}%)")

print(f"   • MM (Miopía Magna):")
for clase, count in predictions_df['MM'].value_counts().items():
    print(f"     - {clase}: {count} estudiantes ({count/len(predictions_df)*100:.1f}%)")

print(f"   • COMBO:")
for clase, count in predictions_df['Combo'].value_counts().items():
    print(f"     - {clase}: {count} estudiantes ({count/len(predictions_df)*100:.1f}%)")

print(f"   • DCOMBO:")
for clase, count in predictions_df['DCombo'].value_counts().items():
    print(f"     - {clase}: {count} estudiantes ({count/len(predictions_df)*100:.1f}%)")

print("\n" + "="*80)
print("✅ Análisis completado exitosamente")
print("="*80)


RESUMEN DEL ANÁLISIS

📊 Dataset:
   • Muestras de entrenamiento: 132
   • Muestras de test: 24
   • Número de características: 48
   • Etiquetas predichas: 4

🎯 Predicciones generadas:
   • M (Miopía):
     - SI: 17 estudiantes (70.8%)
     - NO: 7 estudiantes (29.2%)
   • MM (Miopía Magna):
     - NO: 22 estudiantes (91.7%)
     - SI: 2 estudiantes (8.3%)
   • COMBO:
     - M: 13 estudiantes (54.2%)
     - C: 7 estudiantes (29.2%)
     - MM: 4 estudiantes (16.7%)
   • DCOMBO:
     - M1: 9 estudiantes (37.5%)
     - C: 6 estudiantes (25.0%)
     - MM: 5 estudiantes (20.8%)
     - M2: 4 estudiantes (16.7%)

✅ Análisis completado exitosamente
